# 📘 부록: 장기기억 선택과 사용자별 대화 관리

## 핵심 개념

**다음 작업에 필요한 정보를 저장하고, 현재 요청에 맞게 찾아 모델에 전달합니다.** Store·Mem0·LLM Wiki의 역할과 사용자·대화를 연결하는 앱 구조를 살펴봅니다.


## 1. Store·Mem0·LLM Wiki는 무엇이 다를까요?

<img src="images/15_store_mem0_llm_wiki.png" width="1050" alt="Store는 정리한 데이터를 저장하고, Mem0는 대화에서 기억을 추출하며, LLM Wiki는 작업 자료를 연결된 Markdown 지식으로 정리합니다.">

- **Store:** 무엇을 기억할지 직접 정하고 저장·조회합니다. 임베딩 검색도 연결할 수 있습니다.
- **Mem0:** 대화를 전달하면 기억을 추출하고 저장·검색합니다. 추출 기준과 수정·삭제는 설계해야 합니다.
- **LLM Wiki:** 회의록·작업 결과를 Markdown 지식으로 정리·연결·갱신하는 방식입니다.

모두 함께 쓸 필요는 없습니다. 같은 정보는 어디에서 관리할지 정합니다. [Store](https://docs.langchain.com/oss/python/langgraph/stores) · [Mem0](https://docs.mem0.ai/core-concepts/memory-operations/add)


## 2. 작업 결과를 Markdown 지식으로 축적합니다

<img src="images/16_llm_wiki_flow.png" width="1050" alt="현재 사용자 요청에 필요한 위키 문서를 검색·읽고 답변이나 작업을 수행한 뒤 검토한 결과를 위키에 반영합니다.">

**현재 작업이 읽을 문서를 결정합니다.** 제안서 작성에는 요구사항·결정 문서를, 결정 이유를 묻는 질문에는 근거 회의록을 읽습니다.

원본은 보존하고 위키에는 출처를 연결합니다. `AGENTS.md` 등에 갱신 규칙을 정하고, 모순·오래된 내용·깨진 링크를 점검합니다. 파일을 모으는 것만으로 자동 관리되지는 않습니다.

LLM Wiki는 특정 라이브러리가 아닌 설계 방식입니다. **임베딩·그래프 DB·Obsidian은 필수가 아닙니다.** [Karpathy의 LLM Wiki 원문](https://gist.github.com/karpathy/442a6bf555914893e9891c11519de94f)


## 3. 사용자의 대화 목록은 앱에서 관리합니다

<img src="images/17_app_user_thread_memory.png" width="1050" alt="앱 관리 DB는 user_id와 thread_id의 소유 관계를 관리합니다. thread_id는 대화 State를, user_id는 사용자 기억을 찾는 데 사용합니다.">

**한 사용자에게 여러 대화가 연결됩니다.** 직접 만든 앱에서는 다음처럼 관리할 수 있습니다.

| 앱 테이블 | 관리할 정보 |
|---|---|
| users | user_id, 표시 이름 |
| conversations | thread_id, 소유 user_id, 제목, 생성 시각 |

새 대화에는 UUID처럼 충돌하지 않는 ID를 발급합니다. **앱이 소유자를 확인해야 하며, ID 문자열만으로 접근 권한이 보장되지는 않습니다.**

별도 DB 서버는 필수가 아닙니다. 같은 SQLite 파일의 별도 테이블이나 기존 앱 DB를 활용할 수 있습니다. Agent Server에서는 thread 메타데이터와 접근 제어로 관리할 수도 있습니다. [Checkpointer](https://docs.langchain.com/oss/python/langgraph/checkpointers) · [사용자별 접근 제어](https://docs.langchain.com/langsmith/resource-auth)


## 4. 질문 하나가 들어오면 어떤 ID를 사용할까요?

<img src="images/18_request_memory_flow.png" width="1050" alt="서버에서 로그인 사용자를 확인하고 앱 DB에서 대화 소유자를 확인합니다. thread_id로 대화 State, user_id로 관련 기억을 찾아 현재 질문과 함께 모델에 전달합니다.">

**대화는 `thread_id`, 사용자 기억은 `user_id`로 연결합니다.** Store에서는 user_id로 namespace를 만들어 조회합니다.

namespace를 `("counseling", user_id)`처럼 만들 수 있으면 **별도 Store ID 매핑은 필요 없습니다.** 외부 기억 ID나 Wiki 위치가 다를 때만 앱에 연결 정보를 저장합니다. 프로젝트 Wiki는 해당 프로젝트에 접근할 수 있는 사용자에게 연결합니다.


## 5. 사용자 기억은 Store 또는 Mem0로 관리합니다

**같은 사용자 기억을 관리한다면 둘 중 하나부터 선택합니다. 두 도구를 함께 써야 하는 것은 아닙니다.**

| 선택 | 적합한 경우 | 직접 설계할 부분 |
|---|---|---|
| **Store** | 저장할 내용·형식·갱신을 직접 제어하고 싶을 때 | 기억 선택·추출·갱신 로직 |
| **Mem0** | 대화에서 기억을 추출하고 검색하는 기능을 활용하고 싶을 때 | 추출 지침·수정·삭제 기준 |

Store도 별도 LLM 코드를 연결하면 기억을 추출할 수 있습니다. **기억 추출 기능까지 제공받을지, 직접 구성할지**가 선택 기준입니다.

**앱 구성 예:** 앱 관리 DB + Checkpointer + **Store 또는 Mem0**.

**앱 관리 DB**에는 `user_id`와 `thread_id`의 연결·대화 제목을, **Checkpointer**에는 각 대화의 State를 저장합니다. **Store 또는 Mem0**는 사용자 기억을 저장·조회합니다.

프로젝트 결정·근거·산출물을 문서로 축적하려면 **LLM Wiki를 추가할 수 있습니다.** Store와 Mem0를 함께 쓰는 경우에도 각각 관리할 정보를 구분하며, 같은 기억을 두 곳에 저장할 필요는 없습니다.
